# 2. Evaluation Set & Results

Continuing from [`01_Architecture_Overview.ipynb`](01_Architecture_Overview.ipynb). Read time: ~8 minutes. No API key required — every layer either runs deterministically in-notebook or reads from a captured fixture committed to the repo.

An LLM-system portfolio project needs a defensible answer to "how good is it?" That answer is supposed to be a measured number, not a vibe. This notebook walks the four-layer scorer that was built for that purpose: **18 hand-crafted gold answers**, **two deterministic gates plus two LLM-judged layers**, and **per-model-tier baselines committed in [`measurements/`](../measurements/)** so the numbers are reproducible.

## The four-layer model

Every prediction is scored through four independent layers, each catching a different class of error:

| Layer | What it catches | Mechanism |
| :--- | :--- | :--- |
| **Shape** | Malformed JSON, missing required fields | Deterministic JSON-schema check |
| **Correctness** | Wrong enum value (wrong tier, wrong action category, wrong direction) | Deterministic allow-list comparison against gold |
| **Mid** | Recommendation prose is too thin to be useful | LLM judge, score ≥ 30 |
| **Rich** | Recommendation lacks structured details (cost, projected state, evidence) | LLM judge ≥ 60 + structural checks |

Mid and Rich are *gated on Correctness passing*. If the recommendation gets the basic enum vocabulary wrong, the judge never runs — there's no point asking the judge to evaluate the prose richness of a wrong answer.

The split between deterministic gates and LLM-judged layers is deliberate: rules catch what rules are good at (structural conformance, enum equality); the judge catches what only an LLM can (semantic richness of prose). Neither does the other's job. Full design: [`docs/eval-set.md`](../docs/eval-set.md).

## The 18 scenarios at a glance

Each scenario in the eval set carries a hand-crafted gold answer. The scenarios span single-tier issues, cross-tier cascades, and three deliberate "no-action" cases where the right answer is restraint.

In [1]:
import json
from pathlib import Path
from collections import Counter

PROJECT_ROOT = Path('..').resolve()
EXPECTATIONS_DIR = PROJECT_ROOT / 'eval-set' / 'expectations'

scenarios = {}
for sid_dir in sorted(EXPECTATIONS_DIR.iterdir()):
    if sid_dir.is_dir():
        gold_path = sid_dir / 'raw_recommendation.json'
        if gold_path.exists():
            scenarios[sid_dir.name] = json.loads(gold_path.read_text())

# Loud failure if any gold answer is missing — silent miscounts erode trust.
assert len(scenarios) == 18, (
    f'Expected 18 gold answers, found {len(scenarios)}. '
    f'Missing: {set(f"{i:02d}" for i in range(1, 19)) - set(scenarios)}'
)
print(f'Total scenarios with gold answers: {len(scenarios)}  (assertion passed)')
print()

finding_types = Counter(s.get('finding_type') for s in scenarios.values())
primary_tiers = Counter(s.get('primary_tier') for s in scenarios.values())
action_cats   = Counter(s.get('action_category') for s in scenarios.values())

print('--- by finding_type ---')
for k, v in finding_types.most_common():
    print(f'  {k or "(none)":25s} {v}')
print()
print('--- by primary_tier ---')
for k, v in primary_tiers.most_common():
    print(f'  {k or "(none)":25s} {v}')
print()
print('--- by action_category ---')
for k, v in action_cats.most_common():
    print(f'  {k or "(none)":35s} {v}')

Total scenarios with gold answers: 18  (assertion passed)

--- by finding_type ---
  issue_found               15
  diagnostic_deferral       2
  no_issue_found            1

--- by primary_tier ---
  compute                   8
  database                  4
  network                   2
  deferred                  2
  (none)                    1
  cache                     1

--- by action_category ---
  rightsizing                         7
  scaling_policy_change               3
  (none)                              3
  query_cache_optimization            2
  load_balancer_reconfiguration       1
  cache_capacity_adjustment           1
  network_topology_change             1


## One worked gold — scenario 08

Same scenario notebook 01 walked. The gold answer is what the agents should produce; the scorer measures how close they got.

In [2]:
gold_08 = scenarios['08']
print('--- gold answer for scenario 08 (taxonomy fields) ---')
for field in ('finding_type', 'primary_tier', 'secondary_tier', 'action_category'):
    print(f'  {field:18s}: {gold_08.get(field)!r}')
print()
print('--- specific_change (first 400 chars) ---')
print(' ', (gold_08.get('specific_change') or '')[:400])

--- gold answer for scenario 08 (taxonomy fields) ---
  finding_type      : 'issue_found'
  primary_tier      : 'database'
  secondary_tier    : 'compute'
  action_category   : 'query_cache_optimization'

--- specific_change (first 400 chars) ---
  Optimize the top 6 slowest SQL queries and add 2 read replicas with read/write splitting. Specifically: (1) Add a composite index (user_id, id) on carts and (cart_id) on cart_items for the carts-by-user query (p95 820ms, 6.05M calls); (2) Add a composite index (warehouse_id, product_id) on inventory for the inventory-warehouse query (p95 680ms, 3.53M calls); (3) Add a composite index (product_id, 


## Score gold-vs-gold: the deterministic gates

Submit the gold answer as a prediction and run Shape + Correctness in-notebook. Both layers are deterministic — no API key, no network. The expected result is PASS / PASS because the gold is the gold; if these fail, the scorer itself is broken.

In [3]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluator.evaluator import Scorer

# Defensive guard: this cell depends on `scenarios` from cell #3.
assert 'gold_08' in dir(), 'Run earlier cells first; gold_08 is not defined.'

scorer = Scorer.from_eval_set_dir(
    PROJECT_ROOT / 'eval-set',
    dataset_examples_dir=PROJECT_ROOT / 'dataset-examples',
    judge=None,  # judge layers will read from a committed fixture in the next cell
)

prediction = dict(gold_08)
result = scorer.score_one('08', prediction)

shape_v = 'PASS' if result['shape'].passed else 'FAIL'
corr_v  = 'PASS' if result['correctness'].passed else 'FAIL'

print('=== Deterministic layers (in-notebook) ===')
print(f'  Shape        {shape_v}')
print(f'  Correctness  {corr_v}')

=== Deterministic layers (in-notebook) ===
  Shape        PASS
  Correctness  PASS


## The LLM-judged layers — Mid and Rich

Mid and Rich call an LLM judge to score the recommendation's prose richness. Calling the judge live from the notebook would mean every reader needs an API key — friction that defeats the "open and run" goal. Instead the verdict is captured once into a committed fixture at `tests/integration/notebook_fixtures/scenario_08_judge.json` and read back here. The fixture carries the actual Anthropic Haiku judge's score (0–100) + rationale text from that capture.

If the fixture is missing (e.g., a brand-new clone), the cell below prints the one-shot command to capture it (~$0.01).

In [4]:
import json

fixture_path = PROJECT_ROOT / 'tests' / 'integration' / 'notebook_fixtures' / 'scenario_08_judge.json'
if not fixture_path.exists():
    print('=== LLM-judged layers (Mid + Rich) ===')
    print('  (fixture not generated yet)')
    print()
    print('  To capture a real judge verdict for this notebook, run:')
    print('    uv run python tests/integration/notebook_fixtures/generate_judge_fixtures.py')
    print('  with ANTHROPIC_API_KEY in .env. One-time cost ~$0.01.')
else:
    fixture = json.loads(fixture_path.read_text())
    print('=== LLM-judged layers (Mid + Rich) ===')
    print(f"  Judge        {fixture['judge_provider']} / {fixture['judge_model']}")
    print(f"  Captured     {fixture['captured_at']}")
    print()

    # Mid and Rich are both driven by ONE judge call. Same score, same
    # rationale — different thresholds (Mid ≥ 30, Rich ≥ 60). Print the
    # verdicts first, then the rationale once at the bottom so the reader
    # doesn't think the duplicated paragraph is a bug.
    score = None
    rationale = ''
    thresholds = {'mid': 30, 'rich': 60}
    for label, key in [('Mid', 'mid'), ('Rich', 'rich')]:
        layer = fixture[key]
        if layer.get('status') != 'scored':
            print(f"  {label:<5s}  ({layer.get('status')})")
            continue
        v = 'PASS' if layer.get('passed') else 'FAIL'
        judge_check = next(
            (c for c in layer.get('checks', []) if c.get('name') == 'judge_richness'),
            None,
        )
        detail = (judge_check or {}).get('detail') or {}
        score = detail.get('score')
        rationale = detail.get('rationale', '') or rationale
        print(f'  {label:<5s}  {v}  (judge score: {score} vs threshold {thresholds[key]})')

    if rationale:
        print()
        print('  Judge rationale (one call drives both layers; same score, same text):')
        preview = rationale[:480] + ('…' if len(rationale) > 480 else '')
        # Soft-wrap the rationale for readability.
        import textwrap
        for line in textwrap.wrap(preview, width=88):
            print(f'    {line}')

=== LLM-judged layers (Mid + Rich) ===
  Judge        anthropic / claude-haiku-4-5-20251001
  Captured     2026-06-05T08:33:56+00:00

  Mid    PASS  (judge score: 100 vs threshold 30)
  Rich   PASS  (judge score: 100 vs threshold 60)

  Judge rationale (one call drives both layers; same score, same text):
    The prediction's `specific_change` prose is substantively identical to the gold answer.
    Both texts specify the exact same six index optimizations with identical table names,
    column compositions, and performance metrics (e.g., "composite index (user_id, id) on
    carts" for the carts-by-user query at p95 820ms with 6.05M calls). Both recommend
    provisioning "2 read replicas (db.r6g.xlarge)" with R/W splitting to route SELECT
    traffic to replicas. Both explicitly state "Do NOT scale…


## Published baselines

The `measurements/` folder publishes per-model-tier baselines from real 18-scenario runs. Below is the orchestrated Opus-everywhere run — the configuration the project recommends as the demo default. Per-app pass/fail rows are included so a reader can verify the headline numbers.

In [5]:
summary_path = PROJECT_ROOT / 'measurements' / 'orchestrated-opus-opus-summary.txt'
if summary_path.exists():
    print(summary_path.read_text())
else:
    print(f'(not committed: {summary_path.relative_to(PROJECT_ROOT)})')
    print('See measurements/README.md for the published baselines.')

# Baseline summary — integration-test-opus-4-6

Configuration
-------------
  apps       : 18

Stage 1 — generation
--------------------
  scored apps   : 18 / 18
  total elapsed : 0.0s  (avg 0.0s / app)

Stage 2 — scoring (four layers)
-------------------------------

  layer         pass  fail   n/a  total
  ------------ ----- ----- ----- ------
  Shape           18     0     0     18
  Correctness     18     0     0     18
  Mid             18     0     0     18
  Rich            18     0     0     18

  all-four-PASS apps : 18 / 18

  pass = correct verdict (explicit PASS, or correct short-circuit on
         no-action scenarios where there's nothing to judge)
  fail = wrong verdict (explicit FAIL)
  n/a  = layer didn't run (Correctness gate failed, or scorer crashed)

Per-app verdicts
----------------

  app       Shape  Correctness  Mid    Rich     elapsed
  --------  -----  -----------  -----  -----   --------
  app-01    PASS   PASS         PASS   PASS           -
  app-02    P

## What the numbers mean

- A row showing `PASS / PASS / PASS / PASS` means the agent produced a recommendation that (a) parses cleanly, (b) names the right tier + action category, (c) writes prose rich enough to act on, and (d) carries the structured fields (cost impact, projected state, evidence anchors) needed for a human to review.
- The eval-set leans toward *false negatives* over *false positives*: a recommendation that gets the right *idea* but picks a non-gold enum value gets Correctness FAIL, and Mid + Rich are then gated to `n/a`. This is by design — a benchmark that grades leniently produces numbers that drift up as models get smarter at sounding right.
- The [`measurements/README.md`](../measurements/README.md) discusses false-negative semantics and points at specific scenarios where the strict-enum rule rejects substantively-correct answers. Worth reading if you want to interpret the numbers honestly.

Compare across model tiers by looking at the other `measurements/orchestrated-*` files. The same evaluator scores all of them, so the numbers are directly comparable.

## What the eval design proves

1. **The eval was built alongside the system, not bolted on after.** Most agent projects ship a model and ask a reader to take quality on faith. This one ships a four-layer scorer that runs in CI and produces a comparable number across model tiers. Building the eval is the harder half of the work.
2. **Deterministic layers do deterministic work; the judge does the judge's work.** Shape and Correctness are JSON-schema + enum-equality — bit-for-bit reproducible. Mid and Rich are LLM-judged because nothing else can score prose richness. Neither layer does the other's job.
3. **False-negative bias over false-positive bias.** A recommendation that gets the right *idea* but picks a non-gold enum fails Correctness. That trade-off is documented in `measurements/README.md` rather than hidden. A benchmark that grades leniently produces numbers that drift up as models get smarter at sounding right.
4. **Baselines are committed, not boasted.** The numbers in `measurements/` come from real runs and are reproducible by anyone with an API key. No cherry-picked single runs.

## Continue the tour

→ **[03 Harness Design](03_Harness_Design_Deep_Dive.ipynb)** — the mechanisms that make the scores meaningful. The four harnesses, the audit-trail substrate, the backward evidence-chain walk.

To re-score your own predictions: `python -m src.evaluator.eval --app-name app-08 --prediction <your-pred.json>`.